In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 70 — Compliance Evidence Collector

## Goal
Build an agent that **gathers policies, logs, and evidence**
into an **audit package** — with **approval gates** before
finalizing the report.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Evidence collection** | Gather docs, logs, configs into a package |
| **Compliance checks** | Verify requirements against evidence |
| **Approval workflow** | Multi-step with human checkpoints |
| **Audit packaging** | Structured, timestamped report |

## Architecture
```
Audit Request
    ↓
1. Load compliance requirements
2. Gather evidence (policies, logs, configs)
3. Check each requirement against evidence
4. Flag gaps / non-compliance
5. Draft audit report
6. [Human approval]
7. Finalize package
```

## Stack
- `LangGraph` + `ChatOllama`
- Custom evidence/compliance tools
- Simulated policy & log data

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from datetime import datetime
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports OK")

## Step 1 — Simulated Compliance Data

We simulate a company's security compliance posture:
policies, system logs, access reviews, and configs.

In [ ]:
COMPLIANCE_REQUIREMENTS = {
    "SOC2": [
        {"id": "CC6.1", "title": "Access Control", "description": "Logical and physical access controls are in place to restrict access to authorized personnel."},
        {"id": "CC6.2", "title": "Authentication", "description": "Multi-factor authentication is required for all system access."},
        {"id": "CC6.3", "title": "Access Reviews", "description": "User access is reviewed quarterly and terminated promptly upon role change."},
        {"id": "CC7.1", "title": "Incident Response", "description": "Security incidents are detected, reported, and responded to within SLA."},
        {"id": "CC7.2", "title": "Vulnerability Management", "description": "Systems are scanned for vulnerabilities monthly and critical patches applied within 48 hours."},
        {"id": "CC8.1", "title": "Change Management", "description": "All production changes follow a documented approval process."},
        {"id": "CC9.1", "title": "Data Encryption", "description": "Data is encrypted at rest (AES-256) and in transit (TLS 1.2+)."},
    ],
}

EVIDENCE_STORE = {
    "access_control_policy.pdf": {
        "type": "policy",
        "content": """ACCESS CONTROL POLICY v3.2\nEffective: 2025-01-01\n\nAll employees must use SSO with MFA for system access.\nService accounts require approval from Security team.\nPrivileged access requires additional justification and quarterly review.\nAccess is provisioned based on role-based access control (RBAC).\nAll access requests logged in ServiceNow.""",
        "last_updated": "2025-01-01",
    },
    "mfa_config.json": {
        "type": "configuration",
        "content": json.dumps({
            "provider": "Okta",
            "mfa_required": True,
            "methods": ["push", "totp", "webauthn"],
            "session_timeout_minutes": 480,
            "enforce_for": ["all_users"],
            "exceptions": [],
        }, indent=2),
        "last_updated": "2025-03-15",
    },
    "access_review_q1_2025.csv": {
        "type": "log",
        "content": """user,role,status,reviewed_by,review_date\nalice,admin,active,security_lead,2025-03-31\nbob,developer,active,team_lead,2025-03-31\ncarol,developer,active,team_lead,2025-03-31\ndave,billing,terminated_on_review,security_lead,2025-03-31\neve_contractor,read_only,expired,security_lead,2025-03-31""",
        "last_updated": "2025-03-31",
    },
    "incident_response_plan.pdf": {
        "type": "policy",
        "content": """INCIDENT RESPONSE PLAN v2.1\nEffective: 2025-02-01\n\nSeverity Levels:\n- P1 (Critical): Response within 15 min, resolve within 4 hours\n- P2 (High): Response within 1 hour, resolve within 24 hours\n- P3 (Medium): Response within 4 hours, resolve within 1 week\n\nAll incidents logged in PagerDuty + Jira.\nPost-incident review required for P1/P2 within 48 hours.\nQuarterly tabletop exercises conducted.""",
        "last_updated": "2025-02-01",
    },
    "vuln_scan_may_2025.txt": {
        "type": "log",
        "content": """VULNERABILITY SCAN REPORT - May 2025\nScanner: Qualys\nScan Date: 2025-05-15\nAssets Scanned: 214\n\nCritical: 0\nHigh: 2 (patched 2025-05-16)\nMedium: 8 (remediation in progress)\nLow: 23\n\nLast critical patch: CVE-2025-1234 applied 2025-05-16 (within 24h SLA)""",
        "last_updated": "2025-05-16",
    },
    "change_management_log.csv": {
        "type": "log",
        "content": """change_id,description,requester,approver,status,date\nCHG-301,Database upgrade,bob,alice,approved,2025-05-01\nCHG-302,API rate limit update,carol,alice,approved,2025-05-10\nCHG-303,Emergency hotfix auth bug,alice,security_lead,emergency_approved,2025-05-15\nCHG-304,UI color scheme update,dave,carol,approved,2025-05-20""",
        "last_updated": "2025-05-20",
    },
    "encryption_config.json": {
        "type": "configuration",
        "content": json.dumps({
            "at_rest": {"algorithm": "AES-256-GCM", "key_management": "AWS KMS"},
            "in_transit": {"protocol": "TLS 1.3", "min_version": "TLS 1.2", "certificate_authority": "Let's Encrypt"},
            "database": {"transparent_encryption": True, "column_encryption": ["ssn", "credit_card"]},
        }, indent=2),
        "last_updated": "2025-04-01",
    },
    # Missing evidence: no access review for Q2 2025 yet (gap!)
}

_audit_package = {"findings": [], "status": "in_progress"}

print(f"Compliance frameworks: {list(COMPLIANCE_REQUIREMENTS.keys())}")
print(f"Evidence documents: {len(EVIDENCE_STORE)}")
for name, doc in EVIDENCE_STORE.items():
    print(f"  [{doc['type']}] {name} (updated: {doc['last_updated']})")

## Step 2 — Define Compliance Tools

In [ ]:
@tool
def get_requirements(framework: str) -> str:
    """Get compliance requirements for a framework (e.g., 'SOC2')."""
    fw = framework.upper()
    if fw not in COMPLIANCE_REQUIREMENTS:
        return f"Framework '{framework}' not found. Available: {list(COMPLIANCE_REQUIREMENTS.keys())}"
    reqs = COMPLIANCE_REQUIREMENTS[fw]
    lines = [f"{fw} Requirements:"]
    for r in reqs:
        lines.append(f"  {r['id']}: {r['title']} — {r['description']}")
    return "\n".join(lines)

@tool
def list_evidence() -> str:
    """List all available evidence documents with type and last update date."""
    lines = ["Available evidence:"]
    for name, doc in EVIDENCE_STORE.items():
        lines.append(f"  [{doc['type']}] {name} (updated: {doc['last_updated']})")
    return "\n".join(lines)

@tool
def read_evidence(document_name: str) -> str:
    """Read the content of an evidence document."""
    if document_name not in EVIDENCE_STORE:
        return f"Document '{document_name}' not found. Use list_evidence() to see available documents."
    doc = EVIDENCE_STORE[document_name]
    return f"Document: {document_name}\nType: {doc['type']}\nLast Updated: {doc['last_updated']}\n\n{doc['content']}"

@tool
def record_finding(requirement_id: str, status: str, evidence_used: str, notes: str) -> str:
    """Record a compliance finding. status: 'compliant', 'non_compliant', or 'partial'. evidence_used: comma-separated document names."""
    valid_statuses = ["compliant", "non_compliant", "partial"]
    if status not in valid_statuses:
        return f"Invalid status. Use one of: {valid_statuses}"
    finding = {
        "requirement_id": requirement_id,
        "status": status,
        "evidence_used": [e.strip() for e in evidence_used.split(",")],
        "notes": notes,
        "timestamp": datetime.now().isoformat(),
    }
    _audit_package["findings"].append(finding)
    icon = "✅" if status == "compliant" else "⚠️" if status == "partial" else "❌"
    return f"{icon} Finding recorded: {requirement_id} — {status}"

@tool
def generate_audit_report() -> str:
    """Generate the final audit report from all recorded findings."""
    findings = _audit_package["findings"]
    if not findings:
        return "No findings recorded yet."
    
    compliant = sum(1 for f in findings if f["status"] == "compliant")
    partial = sum(1 for f in findings if f["status"] == "partial")
    non_compliant = sum(1 for f in findings if f["status"] == "non_compliant")
    
    report = f"""COMPLIANCE AUDIT REPORT
{'='*60}
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}
Framework: SOC2
Total Requirements: {len(findings)}

SUMMARY:
  ✅ Compliant:     {compliant}
  ⚠️  Partial:       {partial}
  ❌ Non-Compliant: {non_compliant}
{'='*60}

DETAILED FINDINGS:
"""
    for f in findings:
        icon = "✅" if f["status"] == "compliant" else "⚠️" if f["status"] == "partial" else "❌"
        report += f"\n{icon} {f['requirement_id']}: {f['status'].upper()}\n"
        report += f"   Evidence: {', '.join(f['evidence_used'])}\n"
        report += f"   Notes: {f['notes']}\n"
    
    if non_compliant > 0 or partial > 0:
        report += f"\n{'='*60}\n⚠️  ACTION REQUIRED: {non_compliant + partial} items need remediation.\n"
    else:
        report += f"\n{'='*60}\n✅ All requirements met. Ready for external audit.\n"
    
    return report

compliance_tools = [get_requirements, list_evidence, read_evidence, record_finding, generate_audit_report]
print(f"Defined {len(compliance_tools)} compliance tools")

## Step 3 — Create the Compliance Agent

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

COMPLIANCE_PROMPT = """You are a compliance evidence collector preparing for a SOC2 audit.

Your workflow:
1. Get the SOC2 requirements using get_requirements('SOC2')
2. List available evidence documents using list_evidence()
3. For EACH requirement:
   a. Read relevant evidence documents
   b. Assess whether the evidence satisfies the requirement
   c. Record a finding (compliant, partial, or non_compliant)
4. Generate the final audit report

Assessment rules:
- 'compliant': Evidence clearly demonstrates the requirement is met
- 'partial': Some evidence exists but gaps remain
- 'non_compliant': No evidence or evidence shows non-compliance

Always cite specific evidence documents and explain your reasoning. 
Note any missing evidence as gaps."""

compliance_agent = create_react_agent(
    model=llm,
    tools=compliance_tools,
    prompt=COMPLIANCE_PROMPT,
)

def run_audit(request: str):
    print(f"\n{'='*70}")
    print(f"AUDIT REQUEST: {request}")
    print(f"{'='*70}")
    result = compliance_agent.invoke({"messages": [{"role": "user", "content": request}]})
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL: {tc['name']}({str(tc['args'])[:120]})")
        elif role == "ToolMessage":
            print(f"📋 RESULT: {msg.content[:500]}")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 AUDITOR:\n{msg.content}")
    return result

print("Compliance agent ready!")

In [ ]:
# Run the full SOC2 compliance audit
run_audit("Conduct a SOC2 compliance audit. Check each requirement against available evidence and record findings.")

In [ ]:
# Generate the final report
run_audit("Generate the final audit report.")

In [ ]:
# Display findings summary
print("\n" + "="*70)
print("AUDIT FINDINGS DETAIL")
print("="*70)
for f in _audit_package["findings"]:
    icon = "✅" if f["status"] == "compliant" else "⚠️" if f["status"] == "partial" else "❌"
    print(f"\n{icon} {f['requirement_id']}: {f['status']}")
    print(f"   Evidence: {f['evidence_used']}")
    print(f"   Notes: {f['notes'][:150]}")

## Step 4 — Approval Gate Pattern

In production, the audit report needs human approval
before submission. Here's the conceptual pattern:

In [ ]:
def approval_gate():
    """Conceptual pattern for human-in-the-loop audit approval."""
    print("\n" + "="*60)
    print("APPROVAL GATE")
    print("="*60)
    
    findings = _audit_package["findings"]
    non_compliant = [f for f in findings if f["status"] != "compliant"]
    
    if non_compliant:
        print(f"\n⚠️  {len(non_compliant)} items need attention before audit submission:")
        for f in non_compliant:
            print(f"  - {f['requirement_id']}: {f['status']}")
        print("\nOptions:")
        print("  1. Submit with remediation plan")
        print("  2. Defer audit until gaps are resolved")
        print("  3. Request exception from compliance officer")
    else:
        print("\n✅ All requirements met. Audit package ready for submission.")
    
    # In production: input() or web UI for approval
    print("\n[In production, this would pause for human approval]")

approval_gate()

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Evidence collection** | Read policies, logs, configs from store |
| **Compliance mapping** | Match requirements to supporting evidence |
| **Gap analysis** | Detect missing or insufficient evidence |
| **Audit packaging** | Structured findings with timestamps |
| **Approval gate** | Human checkpoint before finalization |

## LangGraph Workflow (Production Pattern)
```
start → load_requirements → gather_evidence → assess_each_requirement
                                                       ↓
                                              record_findings
                                                       ↓
                                              draft_report
                                                       ↓
                                              [interrupt: human approval]
                                                       ↓
                                              finalize_report → end
```

## Extending for Real Audits
- Connect to cloud provider APIs (AWS Config, Azure Policy)
- Integrate with GRC platforms (Vanta, Drata, Tugboat Logic)
- Add screenshot capture for UI-based evidence
- Automate evidence refresh on a schedule
- Add evidence chain-of-custody tracking